# Number needed to treat (NNT) by initiation cohort

This notebook calculates the number needed to treat (NNT) to avert one case of
Alzheimer's dementia, for simulants who start treatment in 2040, 2050, or 2060,
over 5-year and 10-year horizons. It uses the model 16.0 production run.

**Definition.** For a cohort of simulants who start treatment in year $y$, and
a horizon of $H$ years:

$$\text{ARR} = \frac{\text{Ctrl\_Case} - \text{Trt\_Case}}{\text{Trt\_Pop}}, \qquad \text{NNT} = \frac{1}{\text{ARR}}$$

- **Trt_Pop:** simulants who start treatment in year $y$ in the treatment
  scenario.
- **Trt_Case:** MCI to AD dementia transitions in this cohort in calendar
  years $y$ to $y + H - 1$, in the treatment scenario. For example, the 5-year
  horizon for the 2040 cohort includes every time step in 2040 to 2044 and
  nothing from 2045.
- **Ctrl_Case:** the same transitions in the same cohort, in the matched
  zero-impact treatment scenario. In that scenario, simulants start treatment,
  but the treatment has no effect.

A case counts as averted if it does not happen within the horizon. This
includes cases that happen after the horizon and simulants who die first.

**Cohort stratification.** The model stratifies
`transition_count_alzheimers_disease_and_other_dementias` by
`treatment_initiation_year`. Its categories are `2040`, `2050`, `2060`, and
`other`. The model sets the category from the year of
`treatment_effect_event_time`, which is the time a simulant enters the
`treatment_effect` state (`map_treatment_initiation_year` in
`vivarium_csu_alzheimers/components/observers.py`). Model 15.0 called this
stratification `nnt` (renamed in commit `99106aa`).
Simulants who start treatment in another year, or who never start treatment,
are in `other`.

**Data sources (model 16.0).**

| Quantity | Observer | Filter |
|---|---|---|
| Trt_Pop | `transition_count_treatment` | `sub_entity == 'waiting_for_treatment_to_treatment_effect'` and `event_year` equal to the cohort year |
| Trt_Case, Ctrl_Case | `transition_count_alzheimers_disease_and_other_dementias` | `sub_entity` is MCI to AD dementia and `treatment_initiation_year` equals the cohort year |

**Uncertainty.** We sum over random seeds, age groups, and sexes within each
draw. We calculate ARR for each draw and summarize it
across draws with the mean and the 2.5th and 97.5th percentiles. We then
invert the ARR summary to get NNT. The lower NNT bound comes from the upper ARR
bound. We do not take quantiles of NNT for each draw, because NNT is unstable
when ARR is near zero.

In [1]:
import numpy as np
import pandas as pd
import pyarrow.dataset as ds
from pathlib import Path
import re
import warnings

pd.set_option('display.width', 250)
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 200)

!date
!whoami

Fri Nov 20 05:48:16 PST 2026


lutzes


## Settings

In [2]:
# Model 16.0 production run (100 seeds of 20,000 simulants, 25 draws,
# 10 locations)
run_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers'
               '/results/model16.0/model_spec/2026_11_18_10_15_08')
results_dir = run_dir / 'results'

# Each testing scenario has a matched zero-impact treatment scenario
SCENARIO_PAIRS = {
    'Slow': ('slow_testing_with_treatment',
             'slow_testing_zero_impact_treatment'),
    'Intermediate': ('intermediate_testing_with_treatment',
                     'intermediate_testing_zero_impact_treatment'),
    'Accelerated': ('accelerated_testing_with_treatment',
                    'accelerated_testing_zero_impact_treatment'),
}

INITIATION_TRANSITION = 'waiting_for_treatment_to_treatment_effect'
DEMENTIA_TRANSITION = ('alzheimers_mild_cognitive_impairment_state'
                       '_to_alzheimers_disease_state')

# Cohort years in the treatment_initiation_year stratification, and horizons in calendar years.
# A horizon of H years for cohort year y includes calendar years y to
# y + H - 1.
COHORT_YEARS = [2040, 2050, 2060]
HORIZONS = [5, 10]

# Final CSVs go in the no_draws folder of the most recent dated results
# folder, next to the other final CSVs
final_csvs_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers'
                      '/results/final_csvs')
results_date = max(
    path.name for path in final_csvs_dir.iterdir()
    if path.is_dir() and re.fullmatch(r'\d{4}_\d{2}_\d{2}', path.name)
)
nnt_output_path = final_csvs_dir / results_date / 'no_draws' / 'nnt.csv'
print(nnt_output_path)

/mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/no_draws/nnt.csv


## Load data

We read only the columns and rows that we need, and then sum over random seeds,
age groups, and sexes. The location comes from the artifact path.

In [3]:
LOCATION_NAMES = {
    'united_states_of_america': 'United States of America',
    'united_kingdom': 'United Kingdom',
}


def location_from_artifact_path(path):
    stem = Path(path).stem
    return LOCATION_NAMES.get(stem, stem.replace('_', ' ').title())


def load_counts(measure, sub_entity, extra_columns=()):
    # Load one observer, filter to one sub_entity, and sum over seeds, age
    # groups, and sexes.
    columns = ['artifact_path', 'scenario', 'input_draw', 'event_year',
               *extra_columns, 'value']
    dataset = ds.dataset(results_dir / measure, format='parquet')
    table = dataset.to_table(columns=columns,
                             filter=ds.field('sub_entity') == sub_entity)
    df = table.to_pandas()
    for col in ['artifact_path', 'scenario', *extra_columns]:
        df[col] = df[col].astype(str)
    df['location'] = df['artifact_path'].map(location_from_artifact_path)
    df['event_year'] = df['event_year'].astype(int)
    group_cols = ['location', 'scenario', 'input_draw', 'event_year',
                  *extra_columns]
    return df.groupby(group_cols, observed=True)['value'].sum().reset_index()


initiations_raw = load_counts('transition_count_treatment',
                              INITIATION_TRANSITION)
dementia_raw = load_counts(
    'transition_count_alzheimers_disease_and_other_dementias',
    DEMENTIA_TRANSITION, extra_columns=['treatment_initiation_year'])

print(len(initiations_raw), 'initiation rows')
print(len(dementia_raw), 'dementia transition rows')
LOCATIONS = sorted(initiations_raw['location'].unique())
DRAWS = sorted(initiations_raw['input_draw'].unique())
print(len(LOCATIONS), 'locations:', LOCATIONS)
print(len(DRAWS), 'draws')
print('treatment_initiation_year categories:',
      sorted(dementia_raw['treatment_initiation_year'].unique()))

138250 initiation rows
553000 dementia transition rows
10 locations: ['Brazil', 'China', 'Germany', 'Israel', 'Japan', 'Spain', 'Sweden', 'Taiwan', 'United Kingdom', 'United States of America']
25 draws
treatment_initiation_year categories: ['2040', '2050', '2060', 'other']


## Checks

1. **Ctrl_Pop equals Trt_Pop.** The same simulants must start treatment in the
   treatment scenario and in the zero-impact scenario. Initiation counts should
   match exactly in every location, draw, and year.
2. **Cohort cases start in the cohort year.** A simulant gets a cohort category
   only when it starts treatment, so no case in a cohort can happen before the
   cohort year. This also checks that the cohort year and the initiation
   `event_year` use the same time.
3. **No cohort cases in the baseline scenario.** Nobody starts treatment in the
   baseline scenario.

In [4]:
# Check 1: initiations match between matched scenarios
init_wide = initiations_raw.pivot_table(
    index=['location', 'input_draw', 'event_year'], columns='scenario',
    values='value', aggfunc='sum', fill_value=0)
for pair_name, (trt_scenario, ctrl_scenario) in SCENARIO_PAIRS.items():
    diff = (init_wide[trt_scenario] - init_wide[ctrl_scenario]).abs().max()
    print(f'{pair_name}: largest difference in initiations = {diff}')
    assert diff == 0, f'Initiations differ in the {pair_name} pair'

Slow: largest difference in initiations = 0.0
Intermediate: largest difference in initiations = 0.0
Accelerated: largest difference in initiations = 0.0


In [5]:
cohort_cases_raw = dementia_raw[
    dementia_raw['treatment_initiation_year'] != 'other'].copy()
cohort_cases_raw['cohort'] = (
    cohort_cases_raw['treatment_initiation_year'].astype(int))

# Check 2: no cohort cases before the cohort year
early = cohort_cases_raw.query('event_year < cohort and value > 0')
print('Cohort cases before the cohort year:', early['value'].sum())
assert early.empty, 'Cohort cases happen before the cohort year'

# First year with cohort cases, for each cohort (expect the cohort year)
print(cohort_cases_raw.query('value > 0').groupby('cohort')['event_year'].min())

# Check 3: no cohort cases in the baseline scenario
baseline_cases = cohort_cases_raw.query("scenario == 'baseline'")['value'].sum()
print('Cohort cases in baseline:', baseline_cases)
assert baseline_cases == 0, 'Cohort cases in the baseline scenario'

Cohort cases before the cohort year: 0.0
cohort
2040    2040
2050    2050
2060    2060
Name: event_year, dtype: int64
Cohort cases in baseline: 0.0


## Build one table per cohort

One row per location, scenario pair, draw, cohort, and calendar year.

- `cohort_size`: initiations in the cohort year, in the treatment scenario
- `trt_case`: cases in the cohort, in the treatment scenario
- `ctrl_case`: cases in the cohort, in the zero-impact scenario
- `averted`: `ctrl_case − trt_case`
- `years_since_start`: calendar year minus cohort year

In [6]:
cohort_cases = (cohort_cases_raw
                .groupby(['location', 'scenario', 'input_draw', 'cohort',
                          'event_year'])['value'].sum())
cohort_sizes = (initiations_raw[initiations_raw['event_year'].isin(COHORT_YEARS)]
                .rename(columns={'event_year': 'cohort'})
                .set_index(['location', 'scenario', 'input_draw', 'cohort'])
                ['value'])

# Full index, so that years with no cases have a zero
last_year = max(COHORT_YEARS) + max(HORIZONS) - 1
rows = [(location, draw, cohort, year)
        for location in LOCATIONS for draw in DRAWS for cohort in COHORT_YEARS
        for year in range(cohort, last_year + 1)]
full_index = pd.MultiIndex.from_tuples(
    rows, names=['location', 'input_draw', 'cohort', 'event_year'])

pieces = []
for pair_name, (trt_scenario, ctrl_scenario) in SCENARIO_PAIRS.items():
    piece = pd.DataFrame({
        'trt_case': cohort_cases.xs(trt_scenario, level='scenario')
                    .reindex(full_index, fill_value=0.0),
        'ctrl_case': cohort_cases.xs(ctrl_scenario, level='scenario')
                     .reindex(full_index, fill_value=0.0),
    })
    piece['cohort_size'] = (
        cohort_sizes.xs(trt_scenario, level='scenario')
        .reindex(piece.index.droplevel('event_year')).to_numpy())
    piece['scenario'] = pair_name
    pieces.append(piece)

cohort_data = (pd.concat(pieces).reset_index()
               .set_index(['location', 'scenario', 'input_draw', 'cohort',
                           'event_year'])
               .sort_index())
cohort_data['averted'] = cohort_data['ctrl_case'] - cohort_data['trt_case']
cohort_data['years_since_start'] = (
    cohort_data.index.get_level_values('event_year')
    - cohort_data.index.get_level_values('cohort'))
cohort_data

trt_case  ctrl_case  cohort_size  averted  years_since_start
location                 scenario    input_draw cohort event_year                                                              
Brazil                   Accelerated 11         2040   2040           179.0      179.0      28456.0      0.0                  0
                                                       2041           800.0      990.0      28456.0    190.0                  1
                                                       2042          1056.0     1595.0      28456.0    539.0                  2
                                                       2043          1262.0     1879.0      28456.0    617.0                  3
                                                       2044          1350.0     2043.0      28456.0    693.0                  4
...                                                                     ...        ...          ...      ...                ...
United States of America Slow        243        2060   2065          1099.0     1853.0      26071.0    754.0                  5
                                                       2066          1077.0     1719.0      26071.0    642.0                  6
                                                       2067          1113.0     1543.0      26071.0    430.0                  7
                                                       2068           982.0     1335.0      26071.0    353.0                  8
                                                       2069           927.0     1133.0      26071.0    206.0                  9

[45000 rows x 5 columns]

### Cohort sizes (initiations in the cohort year, summed over draws)

In [7]:
# cohort_size repeats on every calendar-year row, so take one row per draw
(cohort_data.groupby(['location', 'scenario', 'input_draw', 'cohort'])
 ['cohort_size'].first()
 .groupby(['location', 'scenario', 'cohort']).sum()
 .unstack('cohort'))

cohort                                     2040       2050       2060
location                 scenario                                    
Brazil                   Accelerated   699549.0   998228.0  1208990.0
                         Intermediate  734136.0   998536.0  1208990.0
                         Slow          546982.0  1000846.0  1209004.0
China                    Accelerated   847897.0   989733.0  1102834.0
                         Intermediate  891474.0   990048.0  1102834.0
                         Slow          664229.0   992367.0  1102849.0
Germany                  Accelerated   431947.0   424813.0   483061.0
                         Intermediate  453223.0   424901.0   483061.0
                         Slow          337801.0   425559.0   483061.0
Israel                   Accelerated   559071.0   834487.0  1001239.0
                         Intermediate  585730.0   834662.0  1001239.0
                         Slow          436441.0   836036.0  1001253.0
Japan                    Accelerated   278336.0   356899.0   390601.0
                         Intermediate  291904.0   356957.0   390601.0
                         Slow          217610.0   357610.0   390601.0
Spain                    Accelerated   476842.0   607850.0   685743.0
                         Intermediate  500909.0   608014.0   685743.0
                         Slow          373584.0   609292.0   685749.0
Sweden                   Accelerated   432715.0   524719.0   612833.0
                         Intermediate  454374.0   524831.0   612833.0
                         Slow          338920.0   525870.0   612837.0
Taiwan                   Accelerated   636963.0   759403.0   878150.0
                         Intermediate  668600.0   759579.0   878150.0
                         Slow          498250.0   761032.0   878164.0
United Kingdom           Accelerated   510165.0   591250.0   680621.0
                         Intermediate  535593.0   591380.0   680621.0
                         Slow          399237.0   592400.0   680628.0
United States of America Accelerated   515975.0   574693.0   673849.0
                         Intermediate  542030.0   574829.0   673849.0
                         Slow          404219.0   575944.0   673857.0

## Sum cases over the horizon

We take the calendar years in the simulation as the horizon. A horizon of $H$
years for cohort year $y$ includes every time step in calendar years $y$ to
$y + H - 1$, and nothing after that. For example, the 5-year horizon for the
2040 cohort includes all cases in 2040 to 2044 and no cases in 2045.

$$\text{averted}(H) = \sum_{Y=y}^{y+H-1} \big(\text{Ctrl\_Case}(Y) - \text{Trt\_Case}(Y)\big)$$

**Note on follow-up time.** The simulation starts on 2022-01-01 and has a time
step of 182 days, so each year has a step in early June and a step in early
December. A simulant in the 2040 cohort starts treatment at one of the two
2040 steps, and its last counted step is in December 2044. So its follow-up is
about 4.5 years if it starts in June 2040 and about 4 years if it starts in
December 2040. This is the follow-up that the calendar-year definition gives.

In [8]:
def averted_within(cohort_data, horizon):
    # Sum cases in calendar years cohort to cohort + horizon - 1 for each
    # location, scenario, draw, and cohort
    in_horizon = cohort_data['years_since_start'] < horizon
    sums = (cohort_data.loc[in_horizon, ['trt_case', 'ctrl_case', 'averted']]
            .groupby(['location', 'scenario', 'input_draw', 'cohort']).sum())
    sums['cohort_size'] = (cohort_data.groupby(
        ['location', 'scenario', 'input_draw', 'cohort'])['cohort_size'].first())
    return sums


by_draw = []
for horizon in HORIZONS:
    sums = averted_within(cohort_data, horizon)
    sums['arr'] = sums['averted'] / sums['cohort_size']
    sums['horizon'] = f'{horizon} years'
    by_draw.append(sums)

by_draw = pd.concat(by_draw).set_index('horizon', append=True).sort_index()
by_draw.head(10)

trt_case  ctrl_case  averted  cohort_size       arr
location scenario    input_draw cohort horizon                                                      
Brazil   Accelerated 11         2040   10 years   11301.0    15233.0   3932.0      28456.0  0.138178
                                       5 years     4647.0     6686.0   2039.0      28456.0  0.071654
                                2050   10 years   16421.0    21997.0   5576.0      41061.0  0.135798
                                       5 years     6674.0     9582.0   2908.0      41061.0  0.070821
                                2060   10 years   20114.0    27100.0   6986.0      50060.0  0.139553
                                       5 years     8165.0    11678.0   3513.0      50060.0  0.070176
                     15         2040   10 years    9413.0    14843.0   5430.0      27900.0  0.194624
                                       5 years     3910.0     6627.0   2717.0      27900.0  0.097384
                                2050   10 years   13525.0    21405.0   7880.0      40248.0  0.195786
                                       5 years     5559.0     9347.0   3788.0      40248.0  0.094116

## Summary functions

In [9]:
def summarize_arr(arr_by_draw, group_cols):
    # Summarize ARR across draws, then invert to get NNT. The lower NNT bound
    # comes from the upper ARR bound. If the lower ARR bound is zero or less,
    # the upper NNT bound is infinite.
    grouped = arr_by_draw.groupby(group_cols)
    summary = pd.DataFrame({
        'arr_mean': grouped.mean(),
        'arr_lower': grouped.quantile(0.025),
        'arr_upper': grouped.quantile(0.975),
    })
    summary['nnt'] = 1 / summary['arr_mean']
    summary['nnt_lower'] = 1 / summary['arr_upper']
    summary['nnt_upper'] = np.where(
        summary['arr_lower'] > 0, 1 / summary['arr_lower'], np.inf)
    return summary


def nnt_table(summary, column_levels):
    # Format NNT as 'mean (lower, upper)' with one column per value of
    # column_levels
    def fmt(row):
        upper = ('inf' if np.isinf(row['nnt_upper'])
                 else f"{row['nnt_upper']:.1f}")
        return f"{row['nnt']:.1f} ({row['nnt_lower']:.1f}, {upper})"
    return summary.apply(fmt, axis=1).unstack(column_levels)

## Results

In [10]:
summary = summarize_arr(
    by_draw['arr'], ['location', 'scenario', 'cohort', 'horizon'])
summary.round(4)

arr_mean  arr_lower  arr_upper      nnt  nnt_lower  nnt_upper
location                 scenario     cohort horizon                                                                
Brazil                   Accelerated  2040   10 years    0.1563     0.1210     0.1933   6.3996     5.1744     8.2629
                                             5 years     0.0777     0.0616     0.0955  12.8690    10.4732    16.2260
                                      2050   10 years    0.1570     0.1228     0.1965   6.3707     5.0879     8.1448
                                             5 years     0.0785     0.0647     0.0951  12.7467    10.5123    15.4534
                                      2060   10 years    0.1564     0.1207     0.1936   6.3947     5.1656     8.2844
                                             5 years     0.0784     0.0645     0.0948  12.7606    10.5432    15.4976
                         Intermediate 2040   10 years    0.1561     0.1228     0.1918   6.4063     5.2150     8.1464
                                             5 years     0.0782     0.0631     0.0951  12.7934    10.5101    15.8397
                                      2050   10 years    0.1569     0.1227     0.1966   6.3720     5.0874     8.1470
                                             5 years     0.0784     0.0647     0.0952  12.7490    10.5083    15.4607
                                      2060   10 years    0.1564     0.1207     0.1936   6.3947     5.1656     8.2844
                                             5 years     0.0784     0.0645     0.0948  12.7606    10.5432    15.4976
                         Slow         2040   10 years    0.1565     0.1231     0.1925   6.3888     5.1954     8.1232
                                             5 years     0.0781     0.0630     0.0949  12.7980    10.5382    15.8637
                                      2050   10 years    0.1571     0.1229     0.1969   6.3666     5.0798     8.1377
                                             5 years     0.0786     0.0648     0.0953  12.7299    10.4903    15.4404
                                      2060   10 years    0.1564     0.1207     0.1936   6.3947     5.1657     8.2844
                                             5 years     0.0784     0.0645     0.0948  12.7603    10.5430    15.4976
China                    Accelerated  2040   10 years    0.1547     0.1207     0.1885   6.4621     5.3050     8.2851
                                             5 years     0.0776     0.0617     0.0946  12.8914    10.5656    16.1985
                                      2050   10 years    0.1540     0.1204     0.1887   6.4927     5.2984     8.3053
                                             5 years     0.0778     0.0638     0.0931  12.8560    10.7414    15.6674
                                      2060   10 years    0.1538     0.1186     0.1865   6.5029     5.3607     8.4294
                                             5 years     0.0778     0.0637     0.0934  12.8526    10.7033    15.7026
                         Intermediate 2040   10 years    0.1546     0.1220     0.1875   6.4688     5.3347     8.1957
                                             5 years     0.0779     0.0631     0.0944  12.8351    10.5941    15.8579
                                      2050   10 years    0.1540     0.1203     0.1887   6.4935     5.2981     8.3094
                                             5 years     0.0778     0.0638     0.0931  12.8580    10.7452    15.6766
                                      2060   10 years    0.1538     0.1186     0.1865   6.5029     5.3607     8.4294
                                             5 years     0.0778     0.0637     0.0934  12.8526    10.7033    15.7026
                         Slow         2040   10 years    0.1548     0.1219     0.1882   6.4593     5.3146     8.2014
                                             5 years     0.0779     0.0624     0.0941  12.8289    10.6304    16.0246
                                      2050   10 years    0.1541     0.1202   

### NNT, mean (95% UI)

In [11]:
nnt_table(summary, ['horizon', 'cohort'])

horizon                                      10 years            5 years        10 years            5 years        10 years            5 years
cohort                                           2040               2040            2050               2050            2060               2060
location                 scenario                                                                                                             
Brazil                   Accelerated   6.4 (5.2, 8.3)  12.9 (10.5, 16.2)  6.4 (5.1, 8.1)  12.7 (10.5, 15.5)  6.4 (5.2, 8.3)  12.8 (10.5, 15.5)
                         Intermediate  6.4 (5.2, 8.1)  12.8 (10.5, 15.8)  6.4 (5.1, 8.1)  12.7 (10.5, 15.5)  6.4 (5.2, 8.3)  12.8 (10.5, 15.5)
                         Slow          6.4 (5.2, 8.1)  12.8 (10.5, 15.9)  6.4 (5.1, 8.1)  12.7 (10.5, 15.4)  6.4 (5.2, 8.3)  12.8 (10.5, 15.5)
China                    Accelerated   6.5 (5.3, 8.3)  12.9 (10.6, 16.2)  6.5 (5.3, 8.3)  12.9 (10.7, 15.7)  6.5 (5.4, 8.4)  12.9 (10.7, 15.7)
                         Intermediate  6.5 (5.3, 8.2)  12.8 (10.6, 15.9)  6.5 (5.3, 8.3)  12.9 (10.7, 15.7)  6.5 (5.4, 8.4)  12.9 (10.7, 15.7)
                         Slow          6.5 (5.3, 8.2)  12.8 (10.6, 16.0)  6.5 (5.3, 8.3)  12.8 (10.8, 15.7)  6.5 (5.4, 8.4)  12.9 (10.7, 15.7)
Germany                  Accelerated   6.3 (5.1, 8.1)  12.8 (10.3, 16.2)  6.3 (5.0, 8.0)  12.7 (10.5, 15.5)  6.4 (5.1, 8.2)  12.7 (10.5, 15.7)
                         Intermediate  6.3 (5.1, 8.1)  12.8 (10.2, 16.0)  6.3 (5.0, 8.0)  12.7 (10.5, 15.5)  6.4 (5.1, 8.2)  12.7 (10.5, 15.7)
                         Slow          6.3 (5.1, 8.0)  12.7 (10.3, 15.8)  6.3 (5.0, 8.0)  12.6 (10.5, 15.5)  6.4 (5.1, 8.2)  12.7 (10.5, 15.7)
Israel                   Accelerated   6.2 (4.9, 8.0)  12.6 (10.2, 15.9)  6.2 (5.0, 7.9)  12.6 (10.4, 15.3)  6.3 (5.0, 8.1)  12.6 (10.4, 15.3)
                         Intermediate  6.2 (5.0, 7.9)  12.6 (10.2, 15.7)  6.2 (5.0, 7.9)  12.6 (10.4, 15.3)  6.3 (5.0, 8.1)  12.6 (10.4, 15.3)
                         Slow          6.2 (5.0, 7.8)  12.6 (10.3, 15.6)  6.2 (5.0, 7.9)  12.6 (10.4, 15.3)  6.3 (5.0, 8.1)  12.6 (10.4, 15.3)
Japan                    Accelerated   6.0 (4.7, 7.8)   12.5 (9.9, 16.2)  6.1 (4.8, 7.7)  12.4 (10.1, 15.1)  6.1 (4.9, 8.0)  12.6 (10.3, 15.7)
                         Intermediate  6.0 (4.8, 7.7)   12.4 (9.7, 15.8)  6.1 (4.8, 7.7)  12.4 (10.1, 15.1)  6.1 (4.9, 8.0)  12.6 (10.3, 15.7)
                         Slow          6.0 (4.8, 7.7)   12.5 (9.9, 15.7)  6.1 (4.8, 7.7)  12.4 (10.2, 15.0)  6.1 (4.9, 8.0)  12.6 (10.3, 15.7)
Spain                    Accelerated   6.1 (4.9, 7.9)  12.5 (10.0, 15.9)  6.2 (4.9, 7.8)  12.5 (10.2, 15.3)  6.2 (5.0, 8.1)  12.6 (10.3, 15.3)
                         Intermediate  6.1 (4.9, 7.8)  12.4 (10.0, 15.7)  6.2 (4.9, 7.8)  12.5 (10.2, 15.3)  6.2 (5.0, 8.1)  12.6 (10.3, 15.3)
                         Slow          6.1 (4.9, 7.7)  12.4 (10.0, 15.4)  6.2 (4.9, 7.8)  12.5 (10.2, 15.2)  6.2 (5.0, 8.1)  12.6 (10.3, 15.3)
Sweden                   Accelerated   6.2 (4.9, 8.0)  12.6 (10.1, 16.1)  6.2 (4.9, 7.9)  12.5 (10.2, 15.5)  6.2 (5.0, 8.2)  12.5 (10.2, 15.5)
                         Intermediate  6.2 (4.9, 7.9)  12.6 (10.0, 15.9)  6.2 (4.9, 7.9)  12.5 (10.2, 15.5)  6.2 (5.0, 8.2)  12.5 (10.2, 15.5)
                         Slow          6.2 (4.9, 7.8)  12.5 (10.1, 15.7)  6.2 (4.9, 7.9)  12.5 (10.2, 15.5)  6.2 (5.0, 8.2)  12.5 (10.2, 15.5)
Taiwan                   Accelerated   6.3 (5.0, 8.1)  12.7 (10.3, 16.1)  6.3 (5.0, 8.0)  12.7 (10.5, 15.5)  6.3 (5.1, 8.1)  12.7 (10.3, 15.3)
                         Intermediate  6.3 (5.0, 8.0)  12.7 (10.3, 15.8)  6.3 (5.0, 8.0)  12.7 (10.5, 15.5)  6.3 (5.1, 8.1)  12.7 (10.3, 15.3)
                         Slow          6.3 (5.0, 8.1)  12.7 (10.3, 16.0)  6.3 (5.0, 8.0)  12.7 (10.5, 15.5)  6.3 (5.1, 8.1)  12.7 (10.3, 15.3)
United Kingdom           Accelerated   6.3 (5.0, 8.2)  12.8 (10.2, 16.4)  6.3 (5.0, 8.1)  12.7 (10.5, 15.7)  6.4 (5.1, 8.3)  12.8 (10.5, 15

### Counts behind the results

Summed over draws: cohort size, cases in each scenario within the horizon, and
averted cases. These are simulation counts, not scaled to the population.

In [12]:
counts = (by_draw[['cohort_size', 'ctrl_case', 'trt_case', 'averted']]
          .groupby(['location', 'scenario', 'cohort', 'horizon']).sum())
counts

cohort_size  ctrl_case  trt_case   averted
location                 scenario     cohort horizon                                             
Brazil                   Accelerated  2040   10 years     699549.0   368447.0  259193.0  109254.0
                                             5 years      699549.0   161766.0  107429.0   54337.0
                                      2050   10 years     998228.0   528654.0  372062.0  156592.0
                                             5 years      998228.0   231006.0  152735.0   78271.0
                                      2060   10 years    1208990.0   639872.0  450823.0  189049.0
                                             5 years     1208990.0   280333.0  185630.0   94703.0
                         Intermediate 2040   10 years     734136.0   387151.0  272635.0  114516.0
                                             5 years      734136.0   170114.0  112765.0   57349.0
                                      2050   10 years     998536.0   528841.0  372233.0  156608.0
                                             5 years      998536.0   231106.0  152824.0   78282.0
                                      2060   10 years    1208990.0   639872.0  450823.0  189049.0
                                             5 years     1208990.0   280333.0  185630.0   94703.0
                         Slow         2040   10 years     546982.0   289086.0  203523.0   85563.0
                                             5 years      546982.0   127099.0   84386.0   42713.0
                                      2050   10 years    1000846.0   530246.0  373154.0  157092.0
                                             5 years     1000846.0   231799.0  153226.0   78573.0
                                      2060   10 years    1209004.0   639881.0  450830.0  189051.0
                                             5 years     1209004.0   280335.0  185629.0   94706.0
China                    Accelerated  2040   10 years     847897.0   439799.0  308527.0  131272.0
                                             5 years      847897.0   194676.0  128872.0   65804.0
                                      2050   10 years     989733.0   511917.0  359409.0  152508.0
                                             5 years      989733.0   227381.0  150388.0   76993.0
                                      2060   10 years    1102834.0   571809.0  402100.0  169709.0
                                             5 years     1102834.0   253159.0  167332.0   85827.0
                         Intermediate 2040   10 years     891474.0   462800.0  324925.0  137875.0
                                             5 years      891474.0   205107.0  135617.0   69490.0
                                      2050   10 years     990048.0   512135.0  359597.0  152538.0
                                             5 years      990048.0   227483.0  150477.0   77006.0
                                      2060   10 years    1102834.0   571809.0  402100.0  169709.0
                                             5 years     1102834.0   253159.0  167332.0   85827.0
                         Slow         2040   10 years     664229.0   345523.0  242634.0  102889.0
                                             5 years      664229.0   153412.0  101604.0   51808.0
                                      2050   10 years     992367.0   513392.0  360452.0  152940.0
                                             5 years      992367.0   228056.0  150810.0   77246.0
                                      2060   10 years    1102849.0   571820.0  402108.0  169712.0
                                             5 years     1102849.0   253163.0  167333.0   85830.0
Germany                  Accelerated  2040   10 years     431947.0   228217.0  160087.0   68130.0
                                             5 years      431947.0    99797.0   66120.0   33677.0
                                      2050   10 years     424813.0   225691.0  158267.0   67424.0
                                     

## Final CSV

One row per location, scenario, cohort, and follow-up period. The columns
match the other final CSVs, with these differences:

- `Year` is the cohort year, which is the year that treatment starts.
- `Follow-up Years` is the horizon in calendar years.
- There are no `Age`, `Sex`, or `Disease Stage` columns, because the NNT sums
  over age groups and sexes.
- `Scenario` is the With Treatment scenario. Each one is compared with the
  No Treatment scenario that has the same testing speed.
- `Mean` is 1 / (mean ARR). `95% UI Lower` is 1 / (97.5th percentile of
  ARR), and `95% UI Upper` is 1 / (2.5th percentile of ARR). The upper bound
  is `inf` when the 2.5th percentile of ARR is zero or less.

There is no draw-level version, because NNT for each draw is unstable when
ARR is near zero.

In [13]:
nnt_csv = summary.reset_index()
nnt_csv = pd.DataFrame({
    'Year': nnt_csv['cohort'],
    'Location': nnt_csv['location'],
    'Scenario': nnt_csv['scenario'] + ' Testing, With Treatment',
    'Measure': 'Number Needed to Treat',
    'Metric': 'Number',
    'Follow-up Years': nnt_csv['horizon'].str.split().str[0].astype(int),
    'Mean': nnt_csv['nnt'],
    '95% UI Lower': nnt_csv['nnt_lower'],
    '95% UI Upper': nnt_csv['nnt_upper'],
}).sort_values(['Location', 'Scenario', 'Year', 'Follow-up Years'],
               ignore_index=True)

if nnt_output_path.exists():
    warnings.warn(f'Overwriting existing file {nnt_output_path}')
nnt_csv.to_csv(nnt_output_path, index=False)
print(len(nnt_csv), 'rows written to', nnt_output_path)
nnt_csv

180 rows written to /mnt/team/simulation_science/pub/models/vivarium_csu_alzheimers/results/final_csvs/2026_11_19/no_draws/nnt.csv


,Year,Location,Scenario,Measure,Metric,Follow-up Years,Mean,95% UI Lower,95% UI Upper
0,2040,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,5,12.869030,10.473175,16.226041
1,2040,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,10,6.399574,5.174371,8.262890
2,2050,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,5,12.746708,10.512339,15.453410
3,2050,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,10,6.370700,5.087917,8.144805
4,2060,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,5,12.760584,10.543229,15.497633
5,2060,Brazil,"Accelerated Testing, With Treatment",Number Needed to Treat,Number,10,6.394712,5.165572,8.284384
6,2040,Brazil,"Intermediate Testing, With Treatment",Number Needed to Treat,Number,5,12.793399,10.510073,15.839660
7,2040,Brazil,"Intermediate Testing, With Treatment",Number Needed to Treat,Number,10,6.406323,5.214985,8.146388
8,2050,Brazil,"Intermediate Testing, With Treatment",Number Needed to Treat,Number,5,12.749012,10.508251,15.460684
9,2050,Brazil,"Intermediate Testing, With Treatment",Number Needed to Treat,Number,10,6.372034,5.087437,8.147048


In [14]:
!date

Fri Nov 20 08:12:07 PST 2026
